# BCI Competition IV 2a — exploratory data analysis

This notebook inspects the local GDF release of BCI Competition IV Dataset 2a. It covers file/split structure, recording metadata, event distributions, representative EEG traces, spectral content, and cue-locked epochs.

Dataset facts used when interpreting the plots:

- 9 subjects (`A01`–`A09`), each represented in the training (`T`) and evaluation (`E`) splits.
- 22 EEG channels and 3 EOG channels sampled at 250 Hz.
- Four motor-imagery classes in training data: left hand, right hand, feet, and tongue.
- Training files expose class-specific cue events (`769`–`772`). Evaluation files use the unknown-cue event (`783`); their labels are not contained in this GDF archive.
- Each recording is expected to contain 288 trials across 6 runs.

In [56]:
from collections import Counter
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
from IPython.display import display

mne.set_log_level("ERROR")
warnings.filterwarnings("ignore", message=".*Channel names are not unique.*")
plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 50)

# Defined by Table 1 in the official Dataset 2a description:
# https://www.bbci.de/competition/iv/desc_2a.pdf
EVENT_LABELS = {
    "768": "Start of trial",
    "769": "Left hand",
    "770": "Right hand",
    "771": "Feet",
    "772": "Tongue",
    "783": "Unknown cue",
    "1023": "Rejected trial",
    "1072": "Eye movements",
    "276": "Eyes open",
    "277": "Eyes closed",
    "32766": "Start of run",
}
CLASS_EVENT_ID = {
    "Left hand": 769,
    "Right hand": 770,
    "Feet": 771,
    "Tongue": 772,
}

def find_repo_root(start: Path | None = None) -> Path:
    """Find the repository root whether run from the repo or notebooks/."""
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not locate the repository root")

REPO_ROOT = find_repo_root()
DATA_DIR = REPO_ROOT / "data" / "benchmarks" / "BCICIV_2a_gdf"
if not DATA_DIR.exists():
    raise FileNotFoundError(f"Dataset directory not found: {DATA_DIR}")

gdf_files = sorted(DATA_DIR.glob("*.gdf"))
if not gdf_files:
    raise FileNotFoundError(f"No .gdf files found in {DATA_DIR}")

print(f"Repository: {REPO_ROOT}")
print(f"Dataset:    {DATA_DIR}")
print(f"GDF files:  {len(gdf_files)}")

Repository: /Users/alessandro/Desktop/code/sigma-nova-assignment
Dataset:    /Users/alessandro/Desktop/code/sigma-nova-assignment/data/benchmarks/BCICIV_2a_gdf
GDF files:  18


## 1. File inventory

The filename encodes the subject and split. The inventory below verifies that every subject has both expected files and gives a quick view of disk usage.

In [57]:
inventory = pd.DataFrame(
    {
        "file": path.name,
        "subject": path.stem[:3],
        "split": "training" if path.stem.endswith("T") else "evaluation",
        "size_mb": path.stat().st_size / 1024**2,
    }
    for path in gdf_files
)
display(inventory.round({"size_mb": 1}))

print(f"Total uncompressed size: {inventory['size_mb'].sum():.1f} MiB")

,file,subject,split,size_mb
0,A01E.gdf,A01,evaluation,32.8
1,A01T.gdf,A01,training,32.1
2,A02E.gdf,A02,evaluation,31.6
3,A02T.gdf,A02,training,32.3
4,A03E.gdf,A03,evaluation,30.9
5,A03T.gdf,A03,training,31.5
6,A04E.gdf,A04,evaluation,31.5
7,A04T.gdf,A04,training,28.7
8,A05E.gdf,A05,evaluation,32.4
9,A05T.gdf,A05,training,32.7


Total uncompressed size: 574.7 MiB


## 2. Recording-level summary

First, inspect the metadata fields exposed by the GDF reader.

Only headers and annotations are read in this section. GDF readers can report overflow-detection warnings for this dataset; MNE converts those segments into annotations.

In [58]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    header_example = mne.io.read_raw_gdf(gdf_files[0], preload=False, verbose="ERROR")

print("MNE raw.info fields")
display(pd.DataFrame([sorted(header_example.info.keys())]))

print("Per-channel metadata fields")
display(pd.DataFrame([sorted(header_example.info["chs"][0].keys())]))

print("GDF reader header fields")
display(pd.DataFrame([sorted(header_example._raw_extras[0].keys())]))

MNE raw.info fields


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35
0,acq_pars,acq_stim,bads,ch_names,chs,comps,ctf_head_t,custom_ref_applied,description,dev_ctf_t,dev_head_t,device_info,dig,events,experimenter,file_id,gantry_angle,helium_info,highpass,hpi_meas,hpi_results,hpi_subsystem,kit_system_id,line_freq,lowpass,meas_date,meas_id,nchan,proc_history,proj_id,proj_name,projs,sfreq,subject_info,utc_offset,xplotter_layout


Per-channel metadata fields


,0,1,2,3,4,5,6,7,8,9,10
0,cal,ch_name,coil_type,coord_frame,kind,loc,logno,range,scanno,unit,unit_mul


GDF reader header fields


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32
0,blob,bytes_tot,cal,ch_names,data_offset,digital_max,dtype_byte,dtype_np,event_sfreq,events,exclude,gnd,highpass,impedance,lowpass,max_samp,meas_id,n_records,n_samps,nchan,notch,nsamples,number,offsets,orig_nchan,physical_max,record_length,ref,sel,stim_channel_idxs,subtype,type,units


In [59]:
def summarize_recording(path: Path) -> dict:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        raw = mne.io.read_raw_gdf(path, preload=False, verbose="ERROR")
    eog_channels = [name for name in raw.ch_names if name.upper().startswith("EOG")]
    raw.set_channel_types({name: "eog" for name in eog_channels})

    event_counts = Counter(raw.annotations.description)
    channel_types = Counter(raw.get_channel_types())
    return {
        "file": path.name,
        "subject": path.stem[:3],
        "split": "training" if path.stem.endswith("T") else "evaluation",
        "sampling_hz": raw.info["sfreq"],
        "duration_min": raw.times[-1] / 60,
        "samples": raw.n_times,
        "channels": len(raw.ch_names),
        "eeg_channels": channel_types.get("eeg", 0),
        "eog_channels": channel_types.get("eog", 0),
        "annotations": len(raw.annotations),
        "class_cues": sum(event_counts.get(str(code), 0) for code in CLASS_EVENT_ID.values()),
        "unknown_cues": event_counts.get("783", 0),
        "rejected_trials": event_counts.get("1023", 0),
        "runs": event_counts.get("32766", 0),
        "bad_segments": sum(desc.lower().startswith("bad") for desc in raw.annotations.description),
    }

recordings = pd.DataFrame(summarize_recording(path) for path in gdf_files)
display(recordings.round({"sampling_hz": 1, "duration_min": 1}))

print("Dataset totals")
display(
    recordings.groupby("split")
    .agg(
        files=("file", "count"),
        duration_hours=("duration_min", lambda values: values.sum() / 60),
        class_cues=("class_cues", "sum"),
        unknown_cues=("unknown_cues", "sum"),
        rejected_trials=("rejected_trials", "sum"),
    )
    .round(2)
)

,file,subject,split,sampling_hz,duration_min,samples,channels,eeg_channels,eog_channels,annotations,class_cues,unknown_cues,rejected_trials,runs,bad_segments
0,A01E.gdf,A01,evaluation,250.0,45.8,687000,25,22,3,595,0,288,7,9,0
1,A01T.gdf,A01,training,250.0,44.8,672528,25,22,3,603,288,0,15,9,0
2,A02E.gdf,A02,evaluation,250.0,44.2,662666,25,22,3,593,0,288,5,9,0
3,A02T.gdf,A02,training,250.0,45.1,677169,25,22,3,606,288,0,18,9,0
4,A03E.gdf,A03,evaluation,250.0,43.3,648775,25,22,3,603,0,288,15,9,0
5,A03T.gdf,A03,training,250.0,44.0,660530,25,22,3,606,288,0,18,9,0
6,A04E.gdf,A04,evaluation,250.0,44.0,660047,25,22,3,648,0,288,60,9,0
7,A04T.gdf,A04,training,250.0,40.1,600915,25,22,3,610,288,0,26,7,0
8,A05E.gdf,A05,evaluation,250.0,45.3,679863,25,22,3,600,0,288,12,9,0
9,A05T.gdf,A05,training,250.0,45.7,686120,25,22,3,614,288,0,26,9,0


Dataset totals


,files,duration_hours,class_cues,unknown_cues,rejected_trials
split,,,,,
evaluation,9,6.71,0,2592,224
training,9,6.67,2592,0,264


## 3. Event and class distributions

The training split contains known class cues. The evaluation split contains event `783`, so class balance cannot be recovered from these files alone. Rejected-trial annotations are shown separately and should be considered during preprocessing.

### Expected event sequence

```text
Recording
├── Initial EOG calibration annotations may appear
│   ├── 276    Eyes open
│   ├── 277    Eyes closed
│   └── 1072   Eye movements
│
└── 6 runs
    ├── 32766  Start of run
    │
    └── 48 trials per run
        ├── t = 0 s       768       Start of trial; fixation cross and warning tone
        ├── t = 2 s       769–772   Class cue in the training split
        │                 or 783    Hidden class cue in the evaluation split
        ├── t = 2–6 s               Motor-imagery period
        ├── t = 6 s                 Implicit trial end
        └── next 1.5–2.5 s          Break before the next trial
```

Event `1023` may additionally mark a trial as rejected because of artifacts. Run and trial ends are implicit in the protocol rather than represented by dedicated event codes.

In [ ]:
event_rows = []
for path in gdf_files:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        raw_header = mne.io.read_raw_gdf(path, preload=False, verbose="ERROR")
    eog_channels = [name for name in raw_header.ch_names if name.upper().startswith("EOG")]
    raw_header.set_channel_types({name: "eog" for name in eog_channels})
    for event_code, count in Counter(raw_header.annotations.description).items():
        event_rows.append(
            {
                "file": path.name,
                "subject": path.stem[:3],
                "event_code": event_code,
                "event": EVENT_LABELS.get(event_code, event_code),
                "count": count,
            }
        )

events_df = pd.DataFrame(event_rows)
event_totals = (
    events_df.groupby(["event_code", "event"], as_index=False)["count"]
    .sum()
    .sort_values("count", ascending=False)
)
display(event_totals)

class_counts = (
    events_df[events_df["event_code"].isin({"769", "770", "771", "772"})]
    .pivot_table(index="subject", columns="event", values="count", aggfunc="sum", fill_value=0)
    .reindex(columns=list(CLASS_EVENT_ID))
)
display(class_counts.assign(total=class_counts.sum(axis=1)))

,event_code,event,count
5,768,Start of trial,5184
10,783,Unknown cue,2592
6,769,Left hand,648
7,770,Right hand,648
8,771,Feet,648
9,772,Tongue,648
0,1023,Rejected trial,488
4,32766,Start of run,160
1,1072,Eye movements,18
2,276,Eyes open,17


event,Left hand,Right hand,Feet,Tongue,total
subject,,,,,
A01,72,72,72,72,288
A02,72,72,72,72,288
A03,72,72,72,72,288
A04,72,72,72,72,288
A05,72,72,72,72,288
A06,72,72,72,72,288
A07,72,72,72,72,288
A08,72,72,72,72,288
A09,72,72,72,72,288


## 4. Inspect one recording from the training split

`A01T` is loaded lazily. The table verifies channel names/types and the event timeline before any signal samples are brought into memory.

In [60]:
example_path = DATA_DIR / "A01T.gdf"
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    raw = mne.io.read_raw_gdf(example_path, preload=False, verbose="ERROR")
eog_channels = [name for name in raw.ch_names if name.upper().startswith("EOG")]
raw.set_channel_types({name: "eog" for name in eog_channels})

channel_table = pd.DataFrame(
    {
        "channel": raw.ch_names,
        "type": raw.get_channel_types(),
        "unit": [channel.get("unit") for channel in raw.info["chs"]],
    }
)
display(channel_table)

annotation_table = pd.DataFrame(
    {
        "onset_s": raw.annotations.onset,
        "duration_s": raw.annotations.duration,
        "event_code": raw.annotations.description,
    }
)
annotation_table["event"] = annotation_table["event_code"].map(EVENT_LABELS).fillna(annotation_table["event_code"])
display(annotation_table.head(25))

print(raw)
print(f"Measurement date: {raw.info['meas_date']}")

,channel,type,unit
0,EEG-Fz,eeg,107
1,EEG-0,eeg,107
2,EEG-1,eeg,107
3,EEG-2,eeg,107
4,EEG-3,eeg,107
5,EEG-4,eeg,107
6,EEG-5,eeg,107
7,EEG-C3,eeg,107
8,EEG-6,eeg,107
9,EEG-Cz,eeg,107


,onset_s,duration_s,event_code,event
0,0.000,0.004,32766,Start of run
1,0.000,118.728,276,Eyes open
2,118.732,0.004,32766,Start of run
3,118.732,81.084,277,Eyes closed
4,199.820,0.004,32766,Start of run
5,199.820,166.248,1072,Eye movements
6,366.072,0.004,32766,Start of run
7,367.472,7.500,768,Start of trial
8,369.472,1.252,772,Tongue
9,375.484,7.500,768,Start of trial


<RawGDF | A01T.gdf, 25 x 672528 (2690.1 s), ~24 KiB, data not loaded>
Measurement date: 2005-01-17 12:00:00+00:00
